In [78]:
from pathlib import Path
import duckdb

cwd = Path.cwd()
root = cwd.parent if cwd.name == 'notebooks' else cwd
db_path = root / "data" / "fdic.duckdb"
assert db_path.exists(), f"[duckdb_err]: path {db_path} not found"

with duckdb.connect(str(db_path), read_only=True) as conn:
    df = conn.sql("SELECT * from fct_bank_quarter_training").df()
    print(df.shape) 
    print(df.head().T)


(681297, 20)
                                                             0  \
cert                                                     32266   
report_date                                2001-03-31 00:00:00   
bank_quarter_id                                 32266_20010331   
failure_in_year                                              0   
total_risk_based_capital_pct                         12.008618   
nonperforming_assets_to_assets_pct                    0.183577   
loan_loss_allowance_to_loans_pct                        0.5272   
roa_pct                                                 0.4986   
net_interest_margin_pct                               1.919209   
efficiency_ratio_pct                                 62.743191   
brokered_deposits_pct                                61.592887   
core_deposits_pct                                    48.645647   
fhlb_advances_to_assets_pct                          21.563464   
construction_concentration_pct                       13.342318 

In [79]:
import pandas as pd

df['report_date'] = pd.to_datetime(df['report_date'])

train_split_dt = pd.Timestamp('2008-12-31')
test_split_dt = pd.Timestamp('2010-01-01')

train_df = df[df['report_date'] <= train_split_dt].copy()
test_df = df[df['report_date'] >= test_split_dt].copy()

print(f"Train dataframe: {train_df.shape}; Test dataframe: {test_df.shape}")

print(df['failure_in_year'].sum())
print(train_df['failure_in_year'].sum())
print(test_df['failure_in_year'].sum())


Train dataframe: (286499, 20); Test dataframe: (362408, 20)
2305
530
1109


In [80]:
feature_columns = [
    'nonperforming_assets_to_assets_pct', 
    'loan_loss_allowance_to_loans_pct', 
    'roa_pct', 
    'net_interest_margin_pct', 
    'efficiency_ratio_pct', 
    'brokered_deposits_pct', 
    'core_deposits_pct', 
    'fhlb_advances_to_assets_pct', 
    'construction_concentration_pct', 
    'leverage_ratio_pct', 
    'core_deposits_pct_4q_chg', 
    'nonperforming_assets_to_assets_pct_4q_chg', 
    'roa_pct_4q_chg', 
    'leverage_ratio_pct_4q_chg'
]

x_train = train_df[feature_columns]
y_train = train_df['failure_in_year']

x_test = test_df[feature_columns]
y_test = test_df['failure_in_year']


x_train_null_cnt = x_train.isnull().sum()
print(x_train_null_cnt)


nonperforming_assets_to_assets_pct               0
loan_loss_allowance_to_loans_pct                 0
roa_pct                                          0
net_interest_margin_pct                          0
efficiency_ratio_pct                             0
brokered_deposits_pct                          682
core_deposits_pct                                0
fhlb_advances_to_assets_pct                    601
construction_concentration_pct                 567
leverage_ratio_pct                             602
core_deposits_pct_4q_chg                     42516
nonperforming_assets_to_assets_pct_4q_chg    42516
roa_pct_4q_chg                               42516
leverage_ratio_pct_4q_chg                    43031
dtype: int64


In [81]:
x_train["is_first_year"] = (x_train["core_deposits_pct_4q_chg"].isna() & 
                            x_train["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_train["roa_pct_4q_chg"].isna() & 
                            x_train["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

x_test["is_first_year"] = (x_test["core_deposits_pct_4q_chg"].isna() & 
                            x_test["nonperforming_assets_to_assets_pct_4q_chg"].isna() & 
                            x_test["roa_pct_4q_chg"].isna() & 
                            x_test["leverage_ratio_pct_4q_chg"].isna()
                            ).astype(int)

In [82]:
medians = x_train[feature_columns].median()

x_train[feature_columns] = x_train[feature_columns].fillna(medians)
x_test[feature_columns] = x_test[feature_columns].fillna(medians)

In [83]:
print(f"x-train nulls: \n{x_train.isnull().sum()}\n")


x-train nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [84]:
print(f"x-test nulls: \n{x_test.isnull().sum()}\n")


x-test nulls: 
nonperforming_assets_to_assets_pct           0
loan_loss_allowance_to_loans_pct             0
roa_pct                                      0
net_interest_margin_pct                      0
efficiency_ratio_pct                         0
brokered_deposits_pct                        0
core_deposits_pct                            0
fhlb_advances_to_assets_pct                  0
construction_concentration_pct               0
leverage_ratio_pct                           0
core_deposits_pct_4q_chg                     0
nonperforming_assets_to_assets_pct_4q_chg    0
roa_pct_4q_chg                               0
leverage_ratio_pct_4q_chg                    0
is_first_year                                0
dtype: int64



In [85]:
print(f"medians: \n{x_train.median(numeric_only=True)}")

medians: 
nonperforming_assets_to_assets_pct            0.459944
loan_loss_allowance_to_loans_pct              1.212504
roa_pct                                       0.993106
net_interest_margin_pct                       4.008031
efficiency_ratio_pct                         65.853659
brokered_deposits_pct                         0.000000
core_deposits_pct                            69.014564
fhlb_advances_to_assets_pct                   1.545738
construction_concentration_pct                4.477114
leverage_ratio_pct                            9.404728
core_deposits_pct_4q_chg                     -0.552769
nonperforming_assets_to_assets_pct_4q_chg     0.000000
roa_pct_4q_chg                               -0.008681
leverage_ratio_pct_4q_chg                     0.030701
is_first_year                                 0.000000
dtype: float64


In [86]:
prc_lims = x_train[feature_columns].quantile([0.001, 0.999])
lower, upper = prc_lims.loc[0.001], prc_lims.loc[0.999]

x_train[feature_columns] = x_train[feature_columns].clip(lower=lower, upper=upper, axis=1)
x_test[feature_columns]  = x_test[feature_columns].clip(lower=lower, upper=upper, axis=1)

In [87]:
mean = x_train[feature_columns].mean()
std = x_train[feature_columns].std()

x_train[feature_columns] = (x_train[feature_columns] - mean) / std
x_test[feature_columns] = (x_test[feature_columns] - mean) / std



In [88]:
print(x_train[feature_columns].agg(['mean', 'std']).round(2).T)


                                           mean  std
nonperforming_assets_to_assets_pct          0.0  1.0
loan_loss_allowance_to_loans_pct            0.0  1.0
roa_pct                                    -0.0  1.0
net_interest_margin_pct                    -0.0  1.0
efficiency_ratio_pct                        0.0  1.0
brokered_deposits_pct                       0.0  1.0
core_deposits_pct                          -0.0  1.0
fhlb_advances_to_assets_pct                 0.0  1.0
construction_concentration_pct             -0.0  1.0
leverage_ratio_pct                          0.0  1.0
core_deposits_pct_4q_chg                    0.0  1.0
nonperforming_assets_to_assets_pct_4q_chg   0.0  1.0
roa_pct_4q_chg                             -0.0  1.0
leverage_ratio_pct_4q_chg                  -0.0  1.0


In [89]:
print(x_test[feature_columns].agg(['mean', 'std']).round(2).T)

                                           mean   std
nonperforming_assets_to_assets_pct         0.43  1.74
loan_loss_allowance_to_loans_pct           0.17  1.07
roa_pct                                   -0.02  0.81
net_interest_margin_pct                   -0.33  0.89
efficiency_ratio_pct                      -0.02  0.55
brokered_deposits_pct                      0.02  0.90
core_deposits_pct                          0.75  0.83
fhlb_advances_to_assets_pct               -0.24  0.70
construction_concentration_pct            -0.22  0.58
leverage_ratio_pct                        -0.00  0.61
core_deposits_pct_4q_chg                   0.24  1.08
nonperforming_assets_to_assets_pct_4q_chg -0.19  1.21
roa_pct_4q_chg                             0.02  0.98
leverage_ratio_pct_4q_chg                  0.09  0.43


In [90]:
test_df['leverage_ratio_alarm'] = (test_df['leverage_ratio_pct'] < 5).astype(int)

In [91]:
caught = ((test_df['leverage_ratio_alarm'] == 1) & (y_test == 1)).sum() 
false_alarm = ((test_df['leverage_ratio_alarm'] == 1) & (y_test == 0)).sum()
missed = ((test_df['leverage_ratio_alarm'] == 0) & (y_test == 1)).sum()
corr_let_through = ((test_df['leverage_ratio_alarm'] == 0) & (y_test == 0)).sum()

print(f"caught: {caught}")
print(f"false alarm: {false_alarm}")
print(f"missed: {missed}")
print(f"correctly let through: {corr_let_through}")

recall = caught / (caught + missed)
precision = caught / (caught + false_alarm)

print(f"recall: {recall}")
print(f"precision: {precision}")

lev_limit = lower['leverage_ratio_pct']
clipped = train_df['leverage_ratio_pct'] < lev_limit

print("lower limit:", lev_limit)
print("rows clipped:", clipped.sum())
print("positives clipped:", (clipped & (train_df['failure_in_year'] == 1)).sum(), "of 530")

caught: 981
false alarm: 2725
missed: 128
correctly let through: 358574
recall: 0.8845807033363391
precision: 0.2647058823529412
lower limit: 4.160022050881054
rows clipped: 287
positives clipped: 85 of 530


In [92]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(x_train, y_train)

prob = model.predict_proba(x_test)[:, 1]

risks = pd.Series(prob, index=x_test.index)

top_rows_idx = risks.nlargest(3706).index

model_alarm = risks.index.isin(top_rows_idx).astype(int)
model_alarm = pd.Series(model_alarm, index=x_test.index)

print(model_alarm.sum())


caught = ((model_alarm == 1) & (y_test == 1)).sum()
missed = ((model_alarm == 0) & (y_test == 1)).sum()
false_alarm = ((model_alarm == 1) & (y_test == 0)).sum()

model_recall = caught / (caught + missed)
model_precision = caught / (caught + false_alarm)

print(f"model caught: {caught}")
print(f"model missed: {missed}")
print(f"model's false alarms: {false_alarm}")
print(f"model's precision: {model_precision}")
print(f"model's recall: {model_recall}")

3706
model caught: 874
model missed: 235
model's false alarms: 2832
model's precision: 0.2358337830545062
model's recall: 0.78809738503156


In [93]:
for f, w in zip(model.feature_names_in_, model.coef_[0]):
    print(f"{f} - {w}")

nonperforming_assets_to_assets_pct - 0.3754562636932734
loan_loss_allowance_to_loans_pct - -0.004658334451962886
roa_pct - -0.29246489825146355
net_interest_margin_pct - 0.10577692940269381
efficiency_ratio_pct - 0.0228492364607375
brokered_deposits_pct - 0.19264101226215527
core_deposits_pct - -0.4671311043639947
fhlb_advances_to_assets_pct - 0.0902450442801879
construction_concentration_pct - 0.2647053213960605
leverage_ratio_pct - -2.6223395675327015
core_deposits_pct_4q_chg - 0.15929745097963452
nonperforming_assets_to_assets_pct_4q_chg - 0.022303303994412958
roa_pct_4q_chg - -0.09001923227569598
leverage_ratio_pct_4q_chg - -0.03369199569301498
is_first_year - -0.15449876497489817


In [94]:
x_train['leverage_below_5_prc'] = (train_df['leverage_ratio_pct'] < 5).astype(int)
x_test['leverage_below_5_prc'] = (test_df['leverage_ratio_pct'] < 5).astype(int)

print(x_train['leverage_below_5_prc'].sum())
print(x_test['leverage_below_5_prc'].sum())

621
3706


In [95]:
model2 = LogisticRegression(max_iter=1000)
model2.fit(x_train, y_train)

risks2 = pd.Series(model2.predict_proba(x_test)[:, 1], index=x_test.index)
top_rows_idx2 = risks2.nlargest(3706).index
alarm2 = pd.Series(x_test.index.isin(top_rows_idx2).astype(int), index=x_test.index)

caught = ((alarm2 == 1) & (y_test == 1)).sum()
missed = ((alarm2 == 0) & (y_test == 1)).sum()
false_alarm = ((alarm2 == 1) & (y_test == 0)).sum()

model_recall = caught / (caught + missed)
model_precision = caught / (caught + false_alarm)

print(f"model caught: {caught}")
print(f"model missed: {missed}")
print(f"model's false alarms: {false_alarm}")
print(f"model's precision: {model_precision}")
print(f"model's recall: {model_recall}")

model caught: 910
model missed: 199
model's false alarms: 2796
model's precision: 0.2455477603885591
model's recall: 0.8205590622182146


In [96]:
for f, w in zip(model2.feature_names_in_, model2.coef_[0]):
    print(f"{f} - {w}")

nonperforming_assets_to_assets_pct - 0.3687036166476732
loan_loss_allowance_to_loans_pct - -0.007579987797479344
roa_pct - -0.26157407928124426
net_interest_margin_pct - 0.07979943821372878
efficiency_ratio_pct - 0.0013091391316132177
brokered_deposits_pct - 0.20400708697368902
core_deposits_pct - -0.45887856906571084
fhlb_advances_to_assets_pct - 0.1103915385168911
construction_concentration_pct - 0.27589149482763375
leverage_ratio_pct - -2.2041821509499915
core_deposits_pct_4q_chg - 0.1552680615388102
nonperforming_assets_to_assets_pct_4q_chg - 0.02066205833998606
roa_pct_4q_chg - -0.1010864472844192
leverage_ratio_pct_4q_chg - -0.00880082806751731
is_first_year - -0.21588171146649732
leverage_below_5_prc - 0.9132792792629068


In [97]:
import numpy as np

with duckdb.connect(str(db_path), read_only=True) as conn:
    exits_df = conn.sql("select cert, exit_date from int_bank_exits").df()

exits_by_cert = exits_df.set_index('cert')['exit_date']
test_df['exit_date'] = pd.to_datetime(test_df['cert'].map(exits_by_cert))
pos = test_df[y_test == 1].copy()

days_until_exit = (pos['exit_date'] - pos['report_date']).dt.days
pos['quarters_before_exit'] = np.ceil(days_until_exit / 91.3).astype(int)

print(pos['quarters_before_exit'].value_counts().sort_index())
print(f"total: {len(pos)}")

quarters_before_exit
1    335
2    294
3    254
4    226
Name: count, dtype: int64
total: 1109


In [98]:
pos['baseline_caught'] = test_df.loc[pos.index, 'leverage_ratio_alarm']
pos['model_caught'] = alarm2.loc[pos.index]

summary = pos.groupby('quarters_before_exit').agg(
    failures = ('cert', 'size'),
    baseline = ('baseline_caught', 'sum'),
    model = ('model_caught', 'sum'),
)

summary['baseline_prc'] = (100 * summary['baseline'] / summary['failures']).round(1)
summary['model_prc'] = (100 * summary['model'] / summary['failures']).round(1)

print(summary)
print(summary[['failures', 'baseline', 'model']].sum())

                      failures  baseline  model  baseline_prc  model_prc
quarters_before_exit                                                    
1                          335       321    295          95.8       88.1
2                          294       275    244          93.5       83.0
3                          254       217    203          85.4       79.9
4                          226       168    168          74.3       74.3
failures    1109
baseline     981
model        910
dtype: int64


In [99]:
hard = test_df['leverage_ratio_pct'].isna() | (test_df['leverage_ratio_pct'] >= 5)
y_hard = y_test[hard]
risks_hard = risks2[hard]

print(f"unfagged rows: {hard.sum()}")
print(f"failures among unflagged rows: {y_hard.sum()}")

unfagged rows: 358702
failures among unflagged rows: 128


In [100]:
n = int(hard.sum() * 0.01)
top_hard = risks_hard.nlargest(n).index
caught_hard = y_hard.loc[top_hard].sum()

print(f"extra visits: {n}")
print(f"failures found: {caught_hard} of {y_hard.sum()} ({100 * caught_hard / y_hard.sum():.1f}%)")
print(f"precision: {100 * caught_hard / n:.2f}%")
print(f"random visits find: {n * y_hard.mean():.1f}")

extra visits: 3587
failures found: 57 of 128 (44.5%)
precision: 1.59%
random visits find: 1.3


In [101]:
rows = []

for share in [0.0025, 0.005, 0.01, 0.02]:
    n = int(hard.sum()* share)
    found = y_hard.loc[risks_hard.nlargest(n).index].sum()
    random_found = n * y_hard.mean()
    rows.append({
        "budget_pct": share * 100,
        "extra_visits": n,
        "found": found,
        "found_pct": round(100 * found / y_hard.sum(), 1),
        "precision_pct": round(100 * found / n, 2),
        "lift": round(found / random_found, 1)
    })

budget_tb = pd.DataFrame(rows)
print(budget_tb)

   budget_pct  extra_visits  found  found_pct  precision_pct   lift
0        0.25           896     36       28.1           4.02  112.6
1        0.50          1793     49       38.3           2.73   76.6
2        1.00          3587     57       44.5           1.59   44.5
3        2.00          7174     65       50.8           0.91   25.4


In [102]:
with duckdb.connect(str(db_path), read_only=True) as conn:
    query = """
        select 
        f.bank_name, f.state, qf.*
        from int_bank_quarter_features qf
        join stg_fdic__financials f on qf.bank_quarter_id = f.bank_quarter_id
        where qf.has_loans = 1
          and qf.report_date = (select max(report_date) from int_bank_quarter_features)
    """
    latest_report_df = conn.sql(query).df()

    print(latest_report_df.shape, latest_report_df['report_date'].max())

(4223, 22) 2026-06-30 00:00:00


Latest quarter comparison

In [103]:
x_last_report = latest_report_df[feature_columns].copy()

trend_cols = [c for c in feature_columns if c.endswith('4q_chg')]

x_last_report['is_first_year'] = x_last_report[trend_cols].isna().all(axis=1).astype(int)

x_last_report[feature_columns] = x_last_report[feature_columns].fillna(medians)
x_last_report[feature_columns] = x_last_report[feature_columns].clip(lower=lower, upper=upper, axis=1)
x_last_report[feature_columns] = (x_last_report[feature_columns] - mean) / std

x_last_report['leverage_below_5_prc'] = (latest_report_df['leverage_ratio_pct'] < 5).astype(int)

x_last_report = x_last_report[model2.feature_names_in_]

latest_report_df['risk'] = model2.predict_proba(x_last_report)[:, 1]
latest_report_df['leverage_ratio_alarm'] = latest_report_df['leverage_ratio_pct'] < 5

print(f"banks scored: {len(latest_report_df)}")
print(f"flagged by leverage: {latest_report_df['leverage_ratio_alarm'].sum()}")

banks scored: 4223
flagged by leverage: 5


In [104]:
safe_today = latest_report_df[~latest_report_df['leverage_ratio_alarm']]
n_extra = round(len(safe_today) * 0.005)
model_pick = safe_today.nlargest(n_extra, 'risk').index
latest_report_df['model_flag'] = latest_report_df.index.isin(model_pick)

contrib = x_last_report * model2.coef_[0]
latest_report_df['top_driver'] = contrib.idxmax(axis=1)

watchlist = (latest_report_df[latest_report_df['leverage_ratio_alarm'] | latest_report_df['model_flag']]
             .sort_values(['leverage_ratio_alarm', 'risk'], ascending=[False, False]))

print(f"rule: {latest_report_df['leverage_ratio_alarm'].sum()} | model adds: {n_extra} | total: {len(watchlist)}")
watchlist[['cert', 'report_date', 'leverage_ratio_pct', 'risk', 'leverage_ratio_alarm', 'top_driver']]

rule: 5 | model adds: 21 | total: 26


,cert,report_date,leverage_ratio_pct,risk,leverage_ratio_alarm,top_driver
1273,8681,2026-06-30,2.952610,0.436766,True,nonperforming_assets_to_assets_pct
3763,33802,2026-06-30,2.059680,0.073844,True,nonperforming_assets_to_assets_pct
3108,25054,2026-06-30,2.461229,0.009425,True,leverage_ratio_pct
2751,18924,2026-06-30,2.691179,0.004825,True,leverage_ratio_pct
3323,28266,2026-06-30,3.609394,0.003867,True,leverage_ratio_pct
27,58590,2026-06-30,5.037013,0.157801,False,nonperforming_assets_to_assets_pct
3348,28480,2026-06-30,5.458505,0.126717,False,nonperforming_assets_to_assets_pct
759,4063,2026-06-30,9.014886,0.053054,False,nonperforming_assets_to_assets_pct
3876,34786,2026-06-30,8.154506,0.030847,False,nonperforming_assets_to_assets_pct
1609,10816,2026-06-30,8.437419,0.027359,False,nonperforming_assets_to_assets_pct


In [106]:
watchlist.to_csv('../reports/watchlist_latest.csv', index=False)